**Implicitly ingest bronze files data into data frames and print schemas for column names and data types**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Ingest Raw JSON (Orders)
df_orders_bronze = spark.read \
    .json("Files/bronze/orders_raw.json")

# 2. Ingest Raw CSV (Returns)
df_returns_bronze = spark.read \
    .option("header", "true") \
    .csv("Files/bronze/returns_raw.csv")

# 3. Ingest Pipe-Delimited TXT (Inventory)
df_inventory_bronze = spark.read \
    .option("header", "true") \
    .option("delimiter", "|") \
    .csv("Files/bronze/inventory_raw.txt")

# Display schemas to inspect automatic type inference
print("=== ORDERS BRONZE SCHEMA (JSON) ===")
df_orders_bronze.printSchema()

print("\n=== RETURNS BRONZE SCHEMA (CSV) ===")
df_returns_bronze.printSchema()

print("\n=== INVENTORY BRONZE SCHEMA (PIPE TXT) ===")
df_inventory_bronze.printSchema()

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 4, Finished, Available, Finished, False)

=== ORDERS BRONZE SCHEMA (JSON) ===
root
 |-- Delivery_Status: string (nullable = true)
 |-- Order_Amount$: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Order_ID: long (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Qty: string (nullable = true)
 |-- cust_id: string (nullable = true)


=== RETURNS BRONZE SCHEMA (CSV) ===
root
 |-- Return_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Return_Date: string (nullable = true)
 |-- Return_Amount: string (nullable = true)
 |-- Return_Reason: string (nullable = true)


=== INVENTORY BRONZE SCHEMA (PIPE TXT) ===
root
 |-- Product_Name: string (nullable = true)
 |-- Stock_Available: string (nullable = true)
 |-- Unit_Cost_USD: string (nullable = true)



**Clean bronze df data & explicitly convert column data types & write them into the Tables as DeltaLake Table**

In [3]:
from pyspark.sql.functions import col, trim, lower, to_date, regexp_replace, when
# 1. Transform raw returns DataFrame using ACTUAL Bronze schema
df_orders_silver = df_orders_bronze \
    .withColumn("Order_ID", col("Order_ID").cast("long")) \
    .withColumn("cust_id", col("cust_id").cast("long")) \
    .withColumn("Qty", col("Qty").cast("integer")) \
    .withColumn("Order_Amount", regexp_replace(col("Order_Amount$"), "\\$", "").cast("double")) \
    .withColumn("Product_Name", lower(trim(col("Product_Name")))) \
    .withColumn("Product_Name", regexp_replace(col("Product_Name"), "\\s+", " ")) \
    .withColumn("Delivery_Status", lower(trim(col("Delivery_Status")))) \
    .withColumn("Delivery_Status", when(col("Delivery_Status") == "delivrd", "delivered").otherwise(col("Delivery_Status"))) \
    .withColumn("Order_Date", to_date(col("Order_Date"), "yyyy-MM-dd")) \
    .drop("Order_Amount$")

# 2. Inspect transformed schema
print("=== RETURNS SILVER SCHEMA ===")
df_orders_silver.printSchema()

# 3. Write out to Lakehouse as a Delta Table
df_orders_silver.write.format("delta").mode("overwrite").saveAsTable("orders_silver")

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 5, Finished, Available, Finished, False)

=== RETURNS SILVER SCHEMA ===
root
 |-- Delivery_Status: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_ID: long (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Qty: integer (nullable = true)
 |-- cust_id: long (nullable = true)
 |-- Order_Amount: double (nullable = true)



In [4]:
from pyspark.sql.functions import col, trim, to_date

# 1. Transform raw returns DataFrame using ACTUAL Bronze schema
df_returns_silver = df_returns_bronze \
    .withColumn("Return_ID", col("Return_ID").cast("long")) \
    .withColumn("Order_ID", col("Order_ID").cast("long")) \
    .withColumn("Product", trim(col("Product"))) \
    .withColumn("Return_Date", to_date(col("Return_Date"), "yyyy-MM-dd")) \
    .withColumn("Return_Amount", col("Return_Amount").cast("double")) \
    .withColumn("Return_Reason", trim(col("Return_Reason")))

# 2. Inspect transformed schema
print("=== RETURNS SILVER SCHEMA ===")
df_returns_silver.printSchema()

# 3. Write out to Lakehouse as a Delta Table
df_returns_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("returns_silver")

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 6, Finished, Available, Finished, False)

=== RETURNS SILVER SCHEMA ===
root
 |-- Return_ID: long (nullable = true)
 |-- Order_ID: long (nullable = true)
 |-- Product: string (nullable = true)
 |-- Return_Date: date (nullable = true)
 |-- Return_Amount: double (nullable = true)
 |-- Return_Reason: string (nullable = true)



In [5]:
from pyspark.sql.functions import col, trim, lower, regexp_replace

df_inventory_silver = df_inventory_bronze \
    .withColumn("Product_Name", lower(trim(col("Product_Name")))) \
    .withColumn("Product_Name", regexp_replace(col("Product_Name"), "\\s+", " ")) \
    .withColumn("Stock_Available", col("Stock_Available").cast("integer")) \
    .withColumn("Unit_Cost_USD", col("Unit_Cost_USD").cast("double"))

# 2. Inspect transformed schema
print("=== RETURNS SILVER SCHEMA ===")
df_inventory_silver.printSchema()

# 3. Write out to Lakehouse as a Delta Table
df_inventory_silver.write.format("delta").mode("overwrite").saveAsTable("inventory_silver")

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 7, Finished, Available, Finished, False)

=== RETURNS SILVER SCHEMA ===
root
 |-- Product_Name: string (nullable = true)
 |-- Stock_Available: integer (nullable = true)
 |-- Unit_Cost_USD: double (nullable = true)



<mark>**Inspect the Transaction History (DESCRIBE HISTORY)**</mark>

_Every time a Delta table is created, updated, merged, or deleted, Delta Lake writes a new commit log entry (000000.json, 000001.json, etc.) in the background._

In [6]:
from delta.tables import DeltaTable

# 1. Load the Delta table metadata using its table name
delta_table = DeltaTable.forName(spark, "orders_silver")

# 2. Get the full transaction history DataFrame
history_df = delta_table.history()

# 3. Select key audit columns
history_df.select(
    "version", 
    "timestamp", 
    "operation", 
    "operationParameters", 
    "engineInfo"
).show(truncate=False)

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 8, Finished, Available, Finished, False)

+-------+-----------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------+
|version|timestamp              |operation                        |operationParameters                                                                                                                                                     |engineInfo                                                   |
+-------+-----------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------+
|6      |2026-08-07 14:03:36.185|CREATE OR REPLACE TABLE AS SELECT|{partitionBy -> [], clusterBy -> [],

<mark>**Inspect Physical Files in OneLake Storage**</mark>

In [7]:
from delta.tables import DeltaTable

# 1. Get the exact physical OneLake URI from the Delta metadata catalog
delta_table = DeltaTable.forName(spark, "orders_silver")

# Calls the Delta Lake detail API to fetch operational metadata (table size, number of files, location URI, format).
# And Extracts the first string value from the PySpark result DataFrame
# into a standard Python string variable (table_location).
table_location = delta_table.detail().select("location").collect()[0][0]

print("=== PHYSICAL ONELAKE LOCATION ===")
print(table_location)

# 2. List the physical files at that exact path
files = mssparkutils.fs.ls(table_location)

print(f"\n=== PHYSICAL CONTENTS OF orders_silver ===")
for file in files:
    print(f"Name: {file.name} | Size: {file.size} bytes | Is Directory: {file.isDir}")

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 9, Finished, Available, Finished, False)

=== PHYSICAL ONELAKE LOCATION ===
abfss://0fb97f49-ee11-4156-8428-e94749bc842a@onelake.dfs.fabric.microsoft.com/a4a729ff-2296-4fb9-a511-a9582e944028/Tables/dbo/orders_silver

=== PHYSICAL CONTENTS OF orders_silver ===
Name: _delta_log | Size: 0 bytes | Is Directory: True
Name: _metadata | Size: 0 bytes | Is Directory: True
Name: part-00000-1400dad8-8320-4d30-b544-79169dd40771-c000.snappy.parquet | Size: 4289 bytes | Is Directory: False
Name: part-00000-64a1b124-edb8-4844-943d-7d61459f6e6b-c000.snappy.parquet | Size: 4289 bytes | Is Directory: False
Name: part-00000-8e5ae0f8-1d14-4dab-a2d4-a6f7111858d3-c000.snappy.parquet | Size: 4289 bytes | Is Directory: False
Name: part-00000-ac2acbce-8572-43cc-a3f6-095ebfc6d73e-c000.snappy.parquet | Size: 4314 bytes | Is Directory: False
Name: part-00000-b4772f34-dbb6-4505-81be-d03b6edcafda-c000.snappy.parquet | Size: 4289 bytes | Is Directory: False
Name: part-00000-b5b240c5-baba-4c5a-8045-20ce1a5a65f5-c000.snappy.parquet | Size: 4314 bytes | Is Di

Every transaction in a Delta table writes a JSON file inside the _delta_log folder (00000000000000000000.json, 00000000000000000001.json, etc.). These JSON files record table schema changes, file additions/deletions, operational metrics, and commit timestamps.

<mark>**Here is how to list those commit files and read their raw contents using PySpark.**</mark>

**List Files Inside _delta_log and Read the First Commit Log**


In [8]:
from delta.tables import DeltaTable

# 1. Fetch table location and build the _delta_log path
delta_table = DeltaTable.forName(spark, "orders_silver")
table_location = delta_table.detail().select("location").collect()[0][0]
delta_log_path = f"{table_location}/_delta_log"

print("=== DELTA LOG DIRECTORY ===")
print(delta_log_path)

# 2. List all JSON transaction logs
log_files = mssparkutils.fs.ls(delta_log_path)

print("\n=== COMMIT LOG FILES ===")
for f in log_files:
    if f.name.endswith(".json"):
        print(f"Commit Log: {f.name} | Size: {f.size} bytes")

# 3. Target the initial commit log file (00000000000000000000.json)
first_commit_path = f"{delta_log_path}/00000000000000000000.json"

# 4. Read the raw JSON log into a PySpark DataFrame
df_log = spark.read.json(first_commit_path)

print("\n=== SCHEMA OF DELTA TRANSACTION LOG ===")
df_log.printSchema()

print("\n=== RAW TRANSACTION LOG DATA ===")
df_log.show(truncate=False)

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 10, Finished, Available, Finished, False)

=== DELTA LOG DIRECTORY ===
abfss://0fb97f49-ee11-4156-8428-e94749bc842a@onelake.dfs.fabric.microsoft.com/a4a729ff-2296-4fb9-a511-a9582e944028/Tables/dbo/orders_silver/_delta_log

=== COMMIT LOG FILES ===
Commit Log: 00000000000000000000.json | Size: 2133 bytes
Commit Log: 00000000000000000001.json | Size: 1454 bytes
Commit Log: 00000000000000000002.json | Size: 1454 bytes
Commit Log: 00000000000000000003.json | Size: 1452 bytes
Commit Log: 00000000000000000004.json | Size: 1452 bytes
Commit Log: 00000000000000000005.json | Size: 1452 bytes
Commit Log: 00000000000000000006.json | Size: 1452 bytes

=== SCHEMA OF DELTA TRANSACTION LOG ===
root
 |-- add: struct (nullable = true)
 |    |-- dataChange: boolean (nullable = true)
 |    |-- modificationTime: long (nullable = true)
 |    |-- path: string (nullable = true)
 |    |-- size: long (nullable = true)
 |    |-- stats: string (nullable = true)
 |-- commitInfo: struct (nullable = true)
 |    |-- engineInfo: string (nullable = true)
 |   

<mark>**Isolate and Inspect Each Action Cleanly**</mark>

In [9]:
# 1. Filter and display the commit metadata (Who/When/Operation)
print("=== 1. COMMIT INFO (Operation & Metadata) ===")
df_log.filter("commitInfo IS NOT NULL").select("commitInfo.*").show(truncate=False)

# 2. Filter and display table schema metadata
print("\n=== 2. TABLE METADATA (Schema & Creation) ===")
df_log.filter("metaData IS NOT NULL").select("metaData.*").show(truncate=False)

# 3. Filter and display added Parquet data file details
print("\n=== 3. ADD FILE DETAILS (Parquet files & Stats) ===")
df_log.filter("add IS NOT NULL").select("add.path", "add.size", "add.dataChange").show(truncate=False)

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 11, Finished, Available, Finished, False)

=== 1. COMMIT INFO (Operation & Metadata) ===
+-------------------------------------------------------------+-------------+--------------+---------------------------------+----------------+--------------------------------------------------------------------------------------------------+-------------+------------------------------------+
|engineInfo                                                   |isBlindAppend|isolationLevel|operation                        |operationMetrics|operationParameters                                                                               |timestamp    |txnId                               |
+-------------------------------------------------------------+-------------+--------------+---------------------------------+----------------+--------------------------------------------------------------------------------------------------+-------------+------------------------------------+
|Apache-Spark/3.5.5.5.4.20260622.2 Delta-Lake/3.2.1.20260609.1|false    

<mark>**Building the Gold Analytical Layer (business reporting and analytics)**</mark>

**create gold_order_analytics Delta Table**

In [10]:
from pyspark.sql.functions import col, coalesce, lit, when

# 1. Read Silver Delta tables
df_orders = spark.table("orders_silver")
df_returns = spark.table("returns_silver")
df_inventory = spark.table("inventory_silver")

# 2. Join datasets and handle missing values
df_gold = df_orders \
    .join(df_returns.select("Order_ID", "Return_ID", "Return_Amount", "Return_Reason"), on="Order_ID", how="left") \
    .join(df_inventory, on="Product_Name", how="left") \
    .withColumn("Is_Returned", when(col("Return_ID").isNotNull(), True).otherwise(False)) \
    .withColumn("Net_Revenue", coalesce(col("Order_Amount"), lit(0.0)) - coalesce(col("Return_Amount"), lit(0.0))) \
    .withColumn("Total_Cost", col("Qty") * coalesce(col("Unit_Cost_USD"), lit(0.0))) \
    .withColumn("Profit", col("Net_Revenue") - col("Total_Cost"))


print("=== STARTING DATA QUALITY ASSERTIONS ===")

# Assertion 1: Verify total row count matches Silver source (10 records)
expected_rows = 10
actual_rows = df_gold.count()
assert actual_rows == expected_rows, f"DATA QUALITY FAILURE: Expected {expected_rows} rows, but found {actual_rows} rows."
print(f"✓ Row Count Check Passed: {actual_rows} rows.")

# Assertion 2: Check for missing Order_ID (Primary Key integrity)
null_order_ids = df_gold.filter(col("Order_ID").isNull()).count()
assert null_order_ids == 0, f"DATA QUALITY FAILURE: Found {null_order_ids} null Order_IDs."
print("✓ Primary Key Check Passed: No null Order_IDs found.")

# Assertion 3: Audit missing financial amounts and raise warning
missing_amounts = df_gold.filter(col("Order_Amount").isNull()).count()
if missing_amounts > 0:
    pct_missing = (missing_amounts / actual_rows) * 100
    print(f"⚠️ DATA QUALITY WARNING: {missing_amounts} orders ({pct_missing:.1f}%) have missing Order_Amount. Defaulted to $0.0 for revenue math.")

# Assertion 4: Verify Profit calculation consistency
invalid_profit_math = df_gold.filter(col("Profit") != (col("Net_Revenue") - col("Total_Cost"))).count()
assert invalid_profit_math == 0, f"DATA QUALITY FAILURE: Found {invalid_profit_math} records with incorrect profit calculations."
print("✓ Financial Math Verification Passed: All Profit fields equal (Net_Revenue - Total_Cost).")

print("=== ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY ===")

# 3. Overwrite Gold Delta Table
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_order_analytics")

# 4. Display all 10 rows
display(df_gold)

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 12, Finished, Available, Finished, False)

=== STARTING DATA QUALITY ASSERTIONS ===
✓ Row Count Check Passed: 10 rows.
✓ Primary Key Check Passed: No null Order_IDs found.
⚠️ DATA QUALITY WARNING: 6 orders (60.0%) have missing Order_Amount. Defaulted to $0.0 for revenue math.
✓ Financial Math Verification Passed: All Profit fields equal (Net_Revenue - Total_Cost).
=== ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY ===


SynapseWidget(Synapse.DataFrame, 0a6818a9-875e-40e6-acb7-84276a3dc4a4)

<mark>**Business Aggregations & Analytics**</mark>

Gold layer deliverable for reporting and dashboards, to write aggregate queries on top of gold_order_analytics to calculate executive KPIs (Total Revenue, Total Cost, Total Profit, and Profit Margin %).

In [11]:
from pyspark.sql.functions import sum, count, round, col, avg

# 1. Compute overall executive KPIs
kpi_df = spark.table("gold_order_analytics").agg(
    count("Order_ID").alias("Total_Orders"),
    sum("Qty").alias("Total_Units_Sold"),
    round(sum("Net_Revenue"), 2).alias("Gross_Net_Revenue"),
    round(sum("Total_Cost"), 2).alias("Gross_Total_Cost"),
    round(sum("Profit"), 2).alias("Gross_Profit"),
    round((sum("Profit") / sum("Net_Revenue")) * 100, 2).alias("Profit_Margin_Pct")
)

print("=== EXECUTIVE KPI SUMMARY ===")
display(kpi_df)

# 2. Compute product profitability breakdown
product_kpi = spark.table("gold_order_analytics") \
    .groupBy("Product_Name") \
    .agg(
        sum("Qty").alias("Units_Sold"),
        round(sum("Net_Revenue"), 2).alias("Product_Revenue"),
        round(sum("Profit"), 2).alias("Product_Profit")
    ) \
    .orderBy(col("Product_Profit").desc())

print("=== PRODUCT PROFITABILITY RANKING ===")
display(product_kpi)

StatementMeta(, 22c19855-c4d8-4194-93ea-63864d460fb0, 13, Finished, Available, Finished, False)

=== EXECUTIVE KPI SUMMARY ===


SynapseWidget(Synapse.DataFrame, fdd4c220-5393-4780-9676-8aff2fe3bc35)

=== PRODUCT PROFITABILITY RANKING ===


SynapseWidget(Synapse.DataFrame, be72a835-4c29-4bd0-bc3d-dc9939c5555d)